# Bakery-Datensatz     

- **Quelle**: Lehrstuhl für Logistik und Supply Chain
- **Zielvariable**: Nachfrage `demand` (skaliert)
- **Frequenz**: täglich
- **Zeitraum**: 30.01.2016 bis 30.04.2019 (1186 Tage)
- **Granularität**: unterste Ebene (item_id, store_id, täglich)
---
- Anzahl eindeutiger **Artikel**: 3
- Anzahl eindeutiger **Filialen**: 32
- Anzahl eindeutiger Kombinationen aus **Artikeln** und **Filialen** (= ID): 95

## 1. Data Collection

Dieses Datenset wurde vom Lehrstuhl für Logistik und Supply Chain bereitgestellt.

## 2. Data Overview & Description

Eine detaillierte Beschreibung finden Sie in der Dokumentation: https://katja19.github.io/multi-period-forecasting/0_data_preprocessing/bakery_data/#2-data-description

### load, head, shape, dtypes

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
# For plotting and saving figures
import pandas as pd
import plotly.graph_objects as go
import os
import json
import plotly.io as pio
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Set display options for pandas for full visibility of DataFrame contents
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# get the path
data_path = Path.cwd().parents[1] / 'data' / 'raw' / 'dataBakery.csv'

# Load the dataset
bakery_df = pd.read_csv(data_path)

In [ ]:
bakery_df.head()

In [ ]:
print(bakery_df.shape)

In [ ]:
# check the dtypes of the columns
# bakery_df.dtypes

### checking date column

In [ ]:
# Ensure 'date' is datetime
bakery_df['date'] = pd.to_datetime(bakery_df['date'])

# Gesamtzeitraum
overall_min, overall_max = bakery_df['date'].agg(['min', 'max'])
print(f"Overall: {overall_min.date()} to {overall_max.date()} ({(overall_max - overall_min).days + 1} days)") # +1 to include both start and end dates

# Funktion für min, max & Dauer pro Label
for label in ['train', 'test']:
    subset = bakery_df[bakery_df['label'] == label]['date']
    if not subset.empty:
        min_d, max_d = subset.min(), subset.max()
        duration = (max_d - min_d).days + 1  # Include both start and end dates!!!
        print(f"{label.capitalize()} horizon: {min_d.date()} to {max_d.date()} ({duration} days)")


In [ ]:
# check if there are any missing dates between the min and max dates
def check_missing_dates(df, date_col):
    all_dates = pd.date_range(start=df[date_col].min(), end=df[date_col].max())
    missing_dates = all_dates.difference(df[date_col])
    return missing_dates
missing_dates = check_missing_dates(bakery_df, 'date')
if not missing_dates.empty:
    print(f"Missing dates found: {missing_dates.tolist()}")
else:
    print("No missing dates found in the dataset.")

### describe, info, unique values

In [ ]:
bakery_df.describe(include='all') # include all makes sure all columns are included in the description

In [ ]:
#bakery_df.info()

In [ ]:
# ckeck for unique values
unique_values = bakery_df.nunique()
print("Unique values in each column:")
#unique_values

### amount of unique items and stores

In [ ]:
# count how many columns start with 'item_' and how many start with 'store_'
item_columns = [col for col in bakery_df.columns if col.startswith('item_')]
store_columns = [col for col in bakery_df.columns if col.startswith('store_')]
print(f"Amount of unique items: {len(item_columns)}")
print(f"Amount of unique stores: {len(store_columns)}")

### Scaling Value ❌

#### Wie wird die Nachfrage (demand) skaliert?

Antwort:

$demand\ (skaliert) = \frac{demand\_original\ (reconstructed)}{scalingValue}$

(gruppiert nach id (item_store Kombination))

In [ ]:
scaling_value = bakery_df['scalingValue'].unique()
#scaling_value

In [ ]:
bakery_df['demand_original_reconstructed'] = bakery_df['scalingValue'] * bakery_df['demand']
print(bakery_df["demand_original_reconstructed"].describe() )

In [ ]:
print(bakery_df["demand"].describe() )

In [ ]:
bakery_df[["demand", "scalingValue", "demand_original_reconstructed"]].head(10)

In [ ]:
bakery_df.groupby("id")["scalingValue"].nunique().value_counts()

#### Wie wurde der scalingValue berechnet?

Antwort: Der scalingValue ist bis auf in vier Fällen identisch zu max(demand_original_reconstructed) je ID.

In [ ]:
agg_df = bakery_df.groupby("id").agg({
    "demand_original_reconstructed": ["max", "mean", "std", "median"],
    "scalingValue": "first"  # Da pro ID konstant
}).reset_index()

# Spalten umbenennen
agg_df.columns = ["id", "max", "mean", "std", "median", "scalingValue"]


In [ ]:
print(agg_df.corr(numeric_only=True)["scalingValue"])


In [ ]:
agg_df["diff_max"] = np.abs(agg_df["scalingValue"] - agg_df["max"])
agg_df["diff_mean"] = np.abs(agg_df["scalingValue"] - agg_df["mean"])
agg_df["diff_median"] = np.abs(agg_df["scalingValue"] - agg_df["median"])

agg_df[["id", "scalingValue", "max", "mean", "median", "diff_max", "diff_mean", "diff_median"]].sort_values("diff_max").head(10)


In [ ]:
# check if diff_max is something other than 0
agg_df[agg_df["diff_max"] != 0].shape[0] == 0  # besst case: all diff_max are 0, meaning scalingValue is equal to max(demand_original_reconstructed) for all IDs, but this is not the case here

In [ ]:
# get all unique values of diff_max
unique_diff_max = agg_df["diff_max"].unique()
unique_diff_max
# count how often each unique value occurs
diff_max_counts = agg_df["diff_max"].value_counts()
print("Counts of unique diff_max values:")
print(diff_max_counts)


In [ ]:
agg_df[agg_df["diff_max"] != 0][["id", "scalingValue", "max", "diff_max"]]


## 3. Data Cleaning

### Missing Values

In [ ]:
# check for missing values in the dataset
#missing_values = bakery_df.isnull().sum()
#print("Missing values in each column:")
#if (missing_values > 0).any():
#    print(missing_values[missing_values > 0])
#else:
#    print("No missing values found.")

In [ ]:
#check for missing values in the dataset
missing = bakery_df.isna().sum() # isna() is more robust than isnull()
print(missing[missing > 0] if (missing > 0).any() else "No missing values found.")

### Duplicates

In [ ]:
#chek for duplicate rows in the dataset
print(f"Duplicate rows found: {bakery_df.duplicated().sum()}" if bakery_df.duplicated().any() else "No duplicate rows found.")

### Outliners

In [ ]:
bakery_df.head()


In [ ]:
# cols that are sutable for boxplots
boxplot_vars = [col for col in bakery_df.columns if bakery_df[col].nunique() > 2 and bakery_df[col].dtype in ['bool', 'int', 'float']]
# exclude columns that are not suitable for boxplots 'weekyear', 'month', 'year', 'yearIndex'
boxplot_vars = [col for col in boxplot_vars if col not in ['weekyear', 'month', 'year', 'dayIndex']]
#boxplot_vars

In [ ]:
# Create boxplot for each variable in boxplot_vars
# Using Plotly for interactive boxplots
df = bakery_df[boxplot_vars].copy()
initial_col = df.columns[0]

fig_box = go.Figure()

for col in df.columns:
    fig_box.add_trace(go.Box(
        y=df[col].tolist(),  # WICHTIG: tolist(), damit keine bdata im JSON entsteht
        name=col,
        visible=(col == initial_col)
    ))

dropdown_buttons = [
    dict(
        label=col,
        method="update",
        args=[
            {"visible": [c == col for c in df.columns]},
            {
                "title": f"Boxplot für {col}",
                "yaxis": {
                    "title": {"text": col},
                    "range": [
                        df[col].min() - (df[col].std() * 0.5),
                        df[col].max() + (df[col].std() * 0.5)
                    ]
                }
            }
        ]
    )
    for col in df.columns
]

fig_box.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=dropdown_buttons,
            x=1.15,
            xanchor='left',
            y=1.1,
            yanchor='top'
        )
    ],
    title=f"Boxplot für {initial_col}",
    yaxis_title=initial_col,
    showlegend=False,
    height=500
)

# Plot anzeigen
fig_box.show()

In [ ]:
def save_boxplots():
    # JSON speichern
    fig_json = pio.to_json(fig_box, pretty=True)
    # Speicherpfad setzen
    output_dir = r"..\..\docs\assets\json\bakery"
    output_file = "boxplot_multiple.json"
    output_path = os.path.join(output_dir, output_file)

    # Sicherstellen, dass der Ordner existiert
    os.makedirs(output_dir, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(fig_json)
        
    print(f"Plotly JSON gespeichert unter {output_path}")
    
# Save the boxplots to JSON
#save_boxplots()

### Inconsistency

- Nur ein Teil der Boolischen Variablen (Columns) wurden binär encoded. Der Einheitlichkeit werden die anderen auch binär encoded.
- Anpassung der datentypen für semantische korrektheit, speicher und performance vorteile, modellverhalten.
- Anpassung, sodass alle label encodedten Variablen starten mit der Zahl 0. (hier: nur bei month Anpassung nötig)

In [ ]:
# Check if all values in the float column are whole numbers
can_convert = (bakery_df['scalingValue'] % 1 == 0).all()
if can_convert:
    print("All values in 'scalingValue' are whole numbers. Safe to convert to int.")
else:
    print("Not all values in 'scalingValue' are whole numbers. Cannot safely convert to int.")
    #show these values

# show all values in 'scalingValue' that are not whole numbers
non_whole_values = bakery_df[bakery_df['scalingValue'] % 1 != 0]['scalingValue'].unique()
if non_whole_values.size > 0:
    print("Values in 'scalingValue' that are not whole numbers:")
    print(non_whole_values)

In [ ]:
# get list of columns that has holiday in their name
holiday_cols = [col for col in bakery_df.columns if 'holiday' in col]
# encode these boolish columns as binary
for col in holiday_cols:
    bakery_df[col] = bakery_df[col].astype(int)  # Convert boolean columns to binary (0/1)

In [ ]:
# dytpe of cols starting with promotion_ from float to int
promotion_cols = [col for col in bakery_df.columns if col.startswith('promotion_')]
for col in promotion_cols:
    bakery_df[col] = bakery_df[col].astype(int)  # Convert float columns to int (0/1)

In [ ]:
# ckeck if col dayIndex float values are allways integers
day_index_col = 'dayIndex'
if bakery_df[day_index_col].apply(lambda x: x.is_integer()).all():
    bakery_df[day_index_col] = bakery_df[day_index_col].astype(int)  # Convert to int if all values are integers
    print(f"Converted '{day_index_col}' to int as all values are integers.")
else:
    print(f"Warning: Not all values in '{day_index_col}' are integers. Some values will remain as float.")

In [ ]:
# change dtype of col weekday and month from float to int
bakery_df['weekday'] = bakery_df['weekday'].astype(int)
bakery_df['month'] = bakery_df['month'].astype(int)

In [ ]:
# unique values of col month
bakery_df['month'].unique()

In [ ]:
# decrease the values of column month by one int number, e.g. 1 becomes 0, 12 becommes 11
bakery_df["month"] = bakery_df["month"].apply(lambda x: x - 1 if pd.notnull(x) else x)
#ckeck if correct
print(bakery_df['month'].unique())

## 4. EDA: Data Visualisation

In [ ]:
# Sicherstellen, dass Zeitgruppen existieren
bakery_df['year'] = bakery_df['date'].dt.year #float  -> int
bakery_df['yearMonth'] = bakery_df['date'].dt.strftime('%Y-%m')
bakery_df['yearWeek'] = bakery_df['date'].dt.strftime('%G-%V')

In [ ]:
# Plot speichern
def save_plotly_figure_2(fig, filename, groupedBy):
    filepath = os.path.join("..", "..", "docs", "assets", "json", "bakery", groupedBy, f"{filename}.json")
    fig_json = pio.to_json(fig)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(json.loads(fig_json), f, indent=4)

### Totoal Demand

In [ ]:
# Helferfunktion zum Erstellen und Speichern eines Linienplots
def create_and_save_line_plot(x_values, y_values, title, xaxis_title, 
                              yaxis_title, filename, groupedBy, 
                              outliers=None,xaxis_range=None):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=x_values, y=y_values, mode='lines', name=title))

    if outliers is not None:
        fig.add_trace(go.Scatter(
            x=outliers.index.strftime('%Y-%m-%d').tolist(),
            y=outliers.values.tolist(),
            mode='markers',
            name='Outliers (IQR method)',
            marker=dict(color='red', size=5)
        ))

    fig.update_layout(
        title=title,
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        autosize=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1)
    )
    fig.update_xaxes(type='date', range=xaxis_range)  # <-- geändert

    fig.show()
    #save_plotly_figure_2(fig, filename, groupedBy)

In [ ]:

# ========== TÄGLICHE AGGREGATION ========== #
daily = bakery_df.groupby('date')['demand_original_reconstructed'].sum()
xaxis_range=['2016-01-01', '2019-07-01']  # <-- geändert

# IQR-basiertes Outlier-Detection
Q1, Q3 = daily.quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = daily[(daily < Q1 - 1.5 * IQR) | (daily > Q3 + 1.5 * IQR)]

create_and_save_line_plot(
    x_values=daily.index.strftime('%Y-%m-%d').tolist(),
    y_values=daily.values.tolist(),
    title='Total demand per day with outliers',
    xaxis_title='Date',
    yaxis_title='Total demand',
    filename='total_demand_day',
    groupedBy='totalDemand',
    outliers=outliers,
    xaxis_range=xaxis_range
)

# ========== WÖCHENTLICHE AGGREGATION ========== #
weekly = bakery_df.groupby('yearWeek')['demand_original_reconstructed'].sum()
weekly.index = pd.to_datetime(weekly.index + '-1', format='%G-%V-%u')

create_and_save_line_plot(
    x_values=weekly.index.strftime('%Y-%m-%d').tolist(),
    y_values=weekly.values.tolist(),
    title='Total demand per week',
    xaxis_title='Week',
    yaxis_title='Total demand',
    filename='total_demand_week',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

# ========== MONATLICHE AGGREGATION ========== #
monthly = bakery_df.groupby('yearMonth')['demand_original_reconstructed'].sum()
monthly.index = pd.to_datetime(monthly.index + '-01', format='%Y-%m-%d')

create_and_save_line_plot(
    x_values=monthly.index.strftime('%Y-%m-%d').tolist(),
    y_values=monthly.values.tolist(),
    title='Total demand per month',
    xaxis_title='Month',
    yaxis_title='Total demand',
    filename='total_demand_month',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

# ========== JÄHRLICHE AGGREGATION ========== #
yearly = bakery_df.groupby(bakery_df['date'].dt.year)['demand_original_reconstructed'].sum()
yearly.index = pd.to_datetime(yearly.index, format='%Y')

create_and_save_line_plot(
    x_values=yearly.index.strftime('%Y').tolist(),
    y_values=yearly.values.tolist(),
    title='Total demand per year',
    xaxis_title='Year',
    yaxis_title='Total demand',
    filename='total_demand_year',
    groupedBy='totalDemand',
    xaxis_range=xaxis_range
)

### Demand per Item

In [ ]:
# Aggregating the total demand per day and item
total_demand_per_item_per_day = bakery_df.groupby(['date', 'item_101', 'item_109', 'item_110'])['demand_original_reconstructed'].sum().reset_index()

# Remove the time from the date (only keep the date part)
total_demand_per_item_per_day['date'] = total_demand_per_item_per_day['date'].dt.date

# Ensure date column is datetime for formatting later
total_demand_per_item_per_day['date'] = pd.to_datetime(total_demand_per_item_per_day['date'], errors='coerce')
total_demand_per_item_per_day.dropna(subset=['date'], inplace=True)

# Create Plotly figure
fig = go.Figure()

# List of items to plot
items = ['item_101', 'item_109', 'item_110']

# Add one trace per item
for item in items:
    filtered = total_demand_per_item_per_day[total_demand_per_item_per_day[item] > 0]
    fig.add_trace(go.Scatter(
        x=filtered['date'].dt.strftime('%Y-%m-%d').tolist(),
        y=filtered['demand_original_reconstructed'].tolist(),
        mode='lines',
        name=item
    ))

# Update layout
fig.update_layout(
    title='Total demand per item per day',
    xaxis_title='Date',
    yaxis_title='Demand',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5
    )
)

# Show and save
fig.show()
#save_plotly_figure_2(fig, 'demand_per_item_day', 'perItem')


### Demand per Store

In [ ]:
# Get all store columns
stores = [col for col in bakery_df.columns if col.startswith('store')]

# Total demand per yearWeek and store
temp2 = bakery_df.groupby(['yearWeek'] + stores)['demand_original_reconstructed'].sum().reset_index()

# Compute average demand per store (where store == 1)
avg_demand_per_store = {
    store: temp2.loc[temp2[store] == 1, 'demand_original_reconstructed'].mean()
    for store in stores
}

# Sort stores by average demand (descending)
sorted_stores = sorted(avg_demand_per_store, key=avg_demand_per_store.get, reverse=True)

# Group demand per store per week
total_demand_per_store_per_yearWeek = bakery_df.groupby(['yearWeek'] + sorted_stores)['demand_original_reconstructed'].sum().reset_index()

# Total demand per store
store_total_demand = {
    store: total_demand_per_store_per_yearWeek.loc[total_demand_per_store_per_yearWeek[store] > 0, 'demand_original_reconstructed'].sum()
    for store in sorted_stores
}

# Min-max values for color scale
min_demand = min(store_total_demand.values())
max_demand = max(store_total_demand.values())

# Define reversed color scale
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_stores))  # Reversed!
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

In [ ]:
# Create figure
fig = go.Figure()

# Plot each store
for i, store in enumerate(sorted_stores):
    df_store = total_demand_per_store_per_yearWeek.copy()
    df_store = df_store[df_store[store] > 0]

    df_store['yearWeek'] = pd.to_datetime(df_store['yearWeek'] + '-1', format='%G-%V-%u')

    df_store.drop(columns=[drop_store for drop_store in sorted_stores if drop_store != store], inplace=True)

    df_store = df_store.dropna(subset=['demand_original_reconstructed'])

    fig.add_trace(go.Scatter(
        x=df_store['yearWeek'].tolist(),
        y=df_store['demand_original_reconstructed'].astype(float).tolist(),  # <-- FIXED LINE
        mode='lines',
        name=store,
        line=dict(color=color_scale[i])
    ))

# Layout
fig.update_layout(
    title='Total demand per store per Week',
    xaxis_title='Week',
    yaxis_title='Demand',
    height=600,
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5
    ),
    coloraxis=dict(colorscale=color_scale_name)
)

# Show and save
fig.show()
#save_plotly_figure_2(fig, 'demand_per_store_yearWeek', 'perStore')

### Demand per Item-Store

In [ ]:
# Gruppierung der Nachfrage pro Store-Item-Kombination pro Jahr-Woche
demand_per_store_item_week_df = bakery_df.groupby(
    ['yearWeek'] + stores + ['item_101', 'item_109', 'item_110']
)['demand_original_reconstructed'].sum().reset_index()

# Prüfung: Beispielhafte Aggregation vergleichen
store_X = 'store_17'
item_X = 'item_101'
week_X = '2017-40'

agg_demand = demand_per_store_item_week_df[
    (demand_per_store_item_week_df[store_X] == 1) &
    (demand_per_store_item_week_df[item_X] == 1) &
    (demand_per_store_item_week_df['yearWeek'] == week_X)
]['demand_original_reconstructed'].sum()

orig_demand = bakery_df[
    (bakery_df[store_X] == 1) &
    (bakery_df[item_X] == 1) &
    (bakery_df['yearWeek'] == week_X)
]['demand_original_reconstructed'].sum()

print(f'Agg df: The total demand: {agg_demand}')
print(f'Orig df: The total demand: {orig_demand}')


In [ ]:
# Neue Spalte zur Identifikation von Store-Item-Kombinationen
demand_per_store_item_week_df['store_item_id'] = demand_per_store_item_week_df[
    stores + ['item_101', 'item_109', 'item_110']
].apply(lambda x: '_'.join(x.index[x == 1]), axis=1)

# Entferne Store- und Item-Spalten zur Reduktion
demand_per_store_item_week_df.drop(
    columns=stores + ['item_101', 'item_109', 'item_110'],
    inplace=True
)

# Durchschnittlicher Demand je Kombination
store_item_combinations = demand_per_store_item_week_df['store_item_id'].unique()
avg_demand_per_store_item_week = {
    store_item: demand_per_store_item_week_df[
        demand_per_store_item_week_df['store_item_id'] == store_item
    ]['demand_original_reconstructed'].mean()
    for store_item in store_item_combinations
}

# Sortierung nach durchschnittlichem Demand
sorted_store_item_combinations_week = sorted(
    avg_demand_per_store_item_week,
    key=avg_demand_per_store_item_week.get,
    reverse=True
)

# Farbskala (invertiert)
color_scale_name = "Turbo"
color_positions = np.linspace(1, 0, len(sorted_store_item_combinations_week))
color_scale = px.colors.sample_colorscale(color_scale_name, color_positions)

In [ ]:
# Plot erstellen
fig = go.Figure()

for i, store_item in enumerate(sorted_store_item_combinations_week):
    df_store_item = demand_per_store_item_week_df[
        demand_per_store_item_week_df['store_item_id'] == store_item
    ].copy()

    # Konvertiere yearWeek zu datetime
    df_store_item['yearWeek'] = pd.to_datetime(
        df_store_item['yearWeek'] + '-1', format='%G-%V-%u'
    )

    # Entferne NaNs, cast to float
    df_store_item = df_store_item.dropna(subset=['demand_original_reconstructed'])

    # Plot-Spur hinzufügen
    fig.add_trace(go.Scatter(
        x=df_store_item['yearWeek'].tolist(),
        y=df_store_item['demand_original_reconstructed'].astype(float).tolist(),
        mode='lines',
        name=store_item,
        line=dict(color=color_scale[i])
    ))

# Layout
fig.update_layout(
    title='Total demand per store-item combination per Week',
    xaxis_title='Week',
    yaxis_title='Demand',
    height=600,
    margin=dict(t=50, b=50, l=50, r=250),
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.05
    ),
    coloraxis=dict(colorscale=color_scale_name)
)

# Plot anzeigen
fig.show()

# Speichern
#save_plotly_figure_2(fig, 'demand_per_store_item_week', 'perItemStore')


In [ ]:
# Plot erstellen
fig = go.Figure()

for i, store_item in enumerate(sorted_store_item_combinations_week):
    df_store_item = demand_per_store_item_week_df[
        demand_per_store_item_week_df['store_item_id'] == store_item
    ].copy()

    # --- NEU: yearWeek -> Datum (Montag der ISO-Woche) -> Monatsresample ---  # <-- geändert
    week_str = df_store_item['yearWeek'].astype(str).str.replace(r'^(\d{4})[-_]?(\d{2})$', r'\1-\2', regex=True)  # <-- geändert
    df_store_item['__weekdate'] = pd.to_datetime(week_str + '-1', format='%G-%V-%u')                               # <-- geändert

    # Monatlich aggregieren (Summe je Monat, fehlende Monate als 0 auffüllen)                                       # <-- geändert
    dfm = (df_store_item
           .set_index('__weekdate')
           .resample('MS')['demand_original_reconstructed']
           .sum()
           .asfreq('MS', fill_value=0)
           .reset_index()
           .rename(columns={'__weekdate': 'yearMonth'}))                                                           # <-- geändert

    # Plot-Spur hinzufügen (monatlich)                                                                              # <-- geändert
    fig.add_trace(go.Scatter(
        x=dfm['yearMonth'].tolist(),                                                                               # <-- geändert
        y=dfm['demand_original_reconstructed'].astype(float).tolist(),                                             # <-- geändert
        mode='lines',
        name=store_item,
        line=dict(color=color_scale[i])
    ))

# Layout
fig.update_layout(
    title='Total demand per store-item combination per Month',
    xaxis_title='Month',
    yaxis_title='Demand',
    height=600,
    margin=dict(t=50, b=50, l=50, r=250),
    legend=dict(orientation='v', yanchor='top', y=1, xanchor='left', x=1.05),
    coloraxis=dict(colorscale=color_scale_name)
)

# Optional: hübsche Halbjahres-Ticks
fig.update_xaxes(type='date', tick0='2011-01-01', dtick='M6', tickformat='%b %Y')  # <-- optional

fig.show()
# save_plotly_figure_2(fig, 'demand_per_store_item_month', 'perItemStore')


### Item Store Kombinationen (Heatmaps)

In [ ]:
# creating store_id and item_id columns from the cols starting with 'item_' and 'store_' where they are 1
bakery_df[['store_id', 'item_id']] = bakery_df['id'].str.split('_', expand=True)
bakery_df['store_id'] = bakery_df['store_id'].astype(float).astype(int)
bakery_df['item_id'] = bakery_df['item_id'].astype(float).astype(int)

# Gesamter Absatz pro Store und Item
pivot_df = bakery_df.groupby(['store_id', 'item_id'])['demand_original_reconstructed'].sum().unstack(fill_value=0)
#pivot_df


In [ ]:
plt.figure(figsize=(5, 8))
ax = sns.heatmap(pivot_df, cmap="YlGnBu", annot=True, fmt=".0f", square=False)  # sicherstellen, dass nicht quadratisch

# Beschriftungen NICHT kippen
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.title("Gesamter Absatz pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(5, 6))
sns.heatmap(pivot_df, cmap="YlGnBu", annot=True, fmt=".0f")
plt.title("Gesamter Absatz pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()

In [ ]:
# relative demand per store and item
rel_demand_pivot_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100  # Prozentuale Verteilung

# plot the relative demand as a heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(rel_demand_pivot_df, cmap="YlGnBu", annot=True, fmt=".1f")
plt.title("Relative Nachfrage pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()

In [ ]:
plt.figure(figsize=(5, 8))
ax = sns.heatmap(rel_demand_pivot_df, cmap="YlGnBu", annot=True, fmt=".0f",square=False)
# Beschriftungen NICHT kippen
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.title("Relative Nachfrage pro Store und Item")
plt.xlabel("Item")
plt.ylabel("Store")
plt.tight_layout()
plt.show()

In [ ]:
bakery_df['week'] = bakery_df['date'].dt.isocalendar().week # this makes week has values like 1, 2, ...
#df['year'] = df['date'].dt.year # this makes year has values

# Wochenbasierte Mittelwerte
weekly_avg_df = bakery_df.groupby(['store_id', 'item_id', 'year', 'week'])['demand_original_reconstructed'].mean().reset_index()

# Dann wieder pivotieren (z. B. über Mittel aller Wochen)
weekly_avg_pivot_df = weekly_avg_df.groupby(['store_id', 'item_id'])['demand_original_reconstructed'].mean().unstack(fill_value=0)

In [ ]:
# plot the pivot_avg as a heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(weekly_avg_pivot_df, cmap="YlGnBu", annot=True, fmt=".0f")
plt.title("Durchschnittlicher Absatz pro Store und Item (Wochenbasis)")
plt.xlabel("Item")
plt.ylabel("Store")
plt.show()

In [ ]:
plt.figure(figsize=(5, 8))
ax = sns.heatmap(weekly_avg_pivot_df, cmap="YlGnBu", annot=True, fmt=".0f",square=False)
# Beschriftungen NICHT kippen
plt.xticks(rotation=0)
plt.yticks(rotation=0)

plt.title("Durchschnittlicher Absatz pro Store und Item (Wochenbasis)")
plt.xlabel("Item")
plt.ylabel("Store")
plt.tight_layout()
plt.show()

## 5. Feature Engineering & Transformation

In [ ]:
bakery_df.columns

### Weitere Zeit Variablen

| Feature     | Encoding              | Zweck                  |
| ----------- | --------------------- | ---------------------- |
| `dayIndex`  | linear                | Zeitlicher Trend       |
| `dayofyear` | sin/cos               | Jährliche Saisonalität |
| `weekday`   | sin/cos               | Wöchentliche Zyklen    |
| `month`     | sin/cos               | Monatliche Muster      |
| `year`      | skaliert (z. B. -min) | Langfristiger Trend    |


In [ ]:
bakery_df[['date', 'weekday', 'month', 'year', 'label', 'dayIndex', 'scalingValue', 'demand', 'id']].head()

In [ ]:
# Sicherstellen, dass 'date' Datetime ist
bakery_df['date'] = pd.to_datetime(bakery_df['date'])

# 1. year_scaled (lineares Trendfeature)
bakery_df['year_scaled'] = bakery_df['year'] - bakery_df['year'].min()

In [ ]:
# 2. dayofyear
bakery_df['dayofyear'] = bakery_df['date'].dt.dayofyear
bakery_df['dayofyear_sin'] = np.sin(2 * np.pi * bakery_df['dayofyear'] / 365.25)  # 365.25 für Schaltjahre
bakery_df['dayofyear_cos'] = np.cos(2 * np.pi * bakery_df['dayofyear'] / 365.25)

In [ ]:
# 3. weekday (0=Mon, 6=Sun) – ist schon in 'weekday' vorhanden
bakery_df['weekday_sin'] = np.sin(2 * np.pi * bakery_df['weekday'] / 7)
bakery_df['weekday_cos'] = np.cos(2 * np.pi * bakery_df['weekday'] / 7)

# 4. month (0–11)
bakery_df['month_sin'] = np.sin(2 * np.pi * bakery_df['month'] / 12)
bakery_df['month_cos'] = np.cos(2 * np.pi * bakery_df['month'] / 12)

In [ ]:
bakery_df.head(2)

In [ ]:
# check for nan values in the dataset
nan_cols = bakery_df.isna().sum()  # isna() is more robust than isnull()
print(nan_cols[nan_cols > 0] if (nan_cols > 0).any() else "No missing values found.")

### Hinzufügen von Lag Features und mehr

In [ ]:
# --- 0) Sortieren, Konstanten ---
bakery_df = bakery_df.sort_values(['id', 'date']).copy()

LAGS = [1, 7, 14, 28]
MAX_LAG = max(LAGS)

# --- 1) Demand-Lags erstellen (gruppenweise) ---
for k in LAGS:
    col = f'demand_lag_{k}'
    bakery_df[col] = bakery_df.groupby('id', group_keys=False)['demand'].shift(k)
    # Flag: lag war ursprünglich NaN (vor jeglichem Füllen)
    bakery_df[f'{col}_was_nan'] = bakery_df[col].isna().astype(int)

# --- 2) Nur aus Vergangenheit füllen (keine Median/Mean-Imputation) ---
lag_cols = [f'demand_lag_{k}' for k in LAGS]
bakery_df[lag_cols] = bakery_df.groupby('id', group_keys=False)[lag_cols].ffill()

# --- 3) Diffs jetzt aus den (nur-FFill) Lags berechnen ---
for k in LAGS:
    col = f'demand_diff_{k}'
    bakery_df[col] = bakery_df['demand'] - bakery_df[f'demand_lag_{k}']
    # Flag für diff: entspricht dem ursprünglichen lag-NaN-Flag
    bakery_df[f'{col}_was_nan'] = bakery_df[f'demand_lag_{k}_was_nan']

# --- 4) Rolling-Mittel & Ratios (falls noch nicht vorhanden) ---
for window in [7, 28]:
    rm_col = f'demand_rolling_mean_{window}'
    bakery_df[rm_col] = (bakery_df
                         .groupby('id', group_keys=False)['demand']
                         .transform(lambda x: x.rolling(window=window, min_periods=1).mean()))
    bakery_df[f'demand_ratio_to_{window}_avg'] = bakery_df['demand'] / (bakery_df[rm_col] + 1e-5)

In [ ]:
# --- 5) Weitere manuelle Lags (nur Vergangenheit), inkl. Wetter (nur bakery) ---
EXTRA_BASES = [
    'demand_rolling_mean_7','demand_rolling_mean_28',
    'demand_ratio_to_7_avg','demand_ratio_to_28_avg',
    'demand__standard_deviation_7','demand__standard_deviation_14','demand__standard_deviation_28',
    'demand__maximum_7','demand__maximum_14','demand__maximum_28',
    'rain','temperature',
]
EXTRA_LAGS = [1, 7]

extra_lag_cols = []
for base in EXTRA_BASES:
    if base not in bakery_df.columns:  # falls einzelne Stats nicht existieren
        continue
    for k in EXTRA_LAGS:
        c = f'{base}_lag_{k}'
        bakery_df[c] = bakery_df.groupby('id', group_keys=False)[base].shift(k)
        bakery_df[f'{c}_was_nan'] = bakery_df[c].isna().astype(int)
        extra_lag_cols.append(c)

# Nur aus Vergangenheit füllen (falls innerhalb der Serie NaNs vorhanden sind)
if extra_lag_cols:
    bakery_df[extra_lag_cols] = bakery_df.groupby('id', group_keys=False)[extra_lag_cols].ffill()

# --- 6) Burn-in droppen: die ersten MAX_LAG Zeilen je Serie ---
bakery_df = (bakery_df
             .groupby('id', group_keys=False)
             .apply(lambda g: g.iloc[MAX_LAG:])
             .reset_index(drop=True))

In [ ]:
# --- 7) Sanity-Checks ---
need_no_nan = lag_cols + [f'demand_diff_{k}' for k in LAGS] + extra_lag_cols
nan_summary = bakery_df[need_no_nan].isna().sum().sort_values(ascending=False)
print("NaNs in Kern-Lag/Diff/Extra-Lag-Features nach Burn-in:")
print(nan_summary[nan_summary > 0] if (nan_summary > 0).any() else "Keine NaNs mehr in den Kernfeatures.")
print("Shape:", bakery_df.shape)

## Speichern des neuen Dataframes 

In [ ]:
# save the new dataframe to a new csv file
bakery_df.to_csv(Path.cwd().parents[1] / 'data' / 'basic_preprocessed' / 'bakery_basic_prepro.csv', index=False)

In [ ]:
#ckeck by loading the new dataframe
prepro_df = pd.read_csv(Path.cwd().parents[1] / 'data' / 'basic_preprocessed' / 'bakery_basic_prepro.csv')
prepro_df.head(2)

In [ ]:
# compare the shape of the preprocessed dataframe bevor and after saving and reloading
print(f"Original dataframe at end shape: {bakery_df.shape}")
print(f"Preprocessed dataframe shape:    {prepro_df.shape}")

In [ ]:
#prepro_df.dtypes

In [ ]:
# compare datetype of all columns in the original and preprocessed dataframe
for col in bakery_df.columns:
    if bakery_df[col].dtype != prepro_df[col].dtype:
        print(f"Column '{col}' has different dtype: {bakery_df[col].dtype} vs {prepro_df[col].dtype}")

# Sesonality Check mit Darts

In [ ]:
# === Seasonality-Check (gründlich, EDA) ===
from pathlib import Path
import pandas as pd
import numpy as np

from darts import TimeSeries
from darts.utils.statistics import check_seasonality
try:
    from darts.utils.missing_values import fill_missing_values
    DARTS_HAS_MV_UTILS = True
except Exception:
    DARTS_HAS_MV_UTILS = False

In [ ]:
Path.cwd().parents[1]

In [ ]:
# ----- Konfig -----
DATASET   = "bakery"                 # <— bzw. "m5"
M_LIST    = [7, 14, 28]              # <— zu prüfende Saisonalitäten
VAL_DAYS  = 28
TEST_DAYS = 28

FILL_MISSING = True
FILL_METHOD  = "auto"

PROJECT_ROOT = Path.cwd().parents[1]
OUT_DIR = PROJECT_ROOT / "outputs" / "eda" / DATASET.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hilfsfunktionen
# TimeSeries: fehlende Werte füllen (sollten keine fehlen, aber sicher ist sicher), da NaiveSeasonal kein NaN akzeptiert
def ts_has_nan(ts: TimeSeries) -> bool:
    try:    return np.isnan(ts.values(copy=False)).any()
    except: return ts.pd_series().isna().any()

def ts_fill_missing(ts: TimeSeries, method="auto"):
    if not FILL_MISSING or not ts_has_nan(ts): return ts
    if DARTS_HAS_MV_UTILS:
        try: return fill_missing_values(ts, fill=method)
        except: pass
    s = ts.pd_series().ffill().bfill()
    return TimeSeries.from_series(s, freq=ts.freq_str if ts.freq is not None else "D")

# globaler Cut für Trainingsdaten (für alle Serien gleich, damit vergleichbar)
def get_global_train_cut(df, val_days, test_days):
    last_date = df["date"].max()
    return last_date - pd.Timedelta(days=val_days + test_days)

# führende Nullen abschneiden (für MASE/RMSSE), damit diese nicht die Denominators verfälschen (sollte nicht der Fall sein)
def trim_leading_zeros(ts: TimeSeries) -> TimeSeries:
    a = ts.values(copy=False).flatten()
    nz = np.nonzero(a)[0]
    return ts if len(nz)==0 else ts[nz[0]:]

In [ ]:
# Da NaiveSeasonal nur target Spalte braucht, alle anderen entfernen
prepro_df = (prepro_df[["date", "id", "demand"]]
      .drop_duplicates(subset=["id","date"])
      .sort_values(["id","date"])
      .reset_index(drop=True))

prepro_df.head(2)

In [ ]:
prepro_df.dtypes

In [ ]:
# change col date to datetime
prepro_df['date'] = pd.to_datetime(prepro_df['date'])
prepro_df.dtypes

In [ ]:
# prüft ob es Lücken in den Zeitreihen gibt
def quality_checks(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for gid, g in df.groupby("id"):
        
        exp = (g['date'].max() - g['date'].min()).days + 1
        actual = g['date'].nunique()
        gaps = exp - actual
        stats.append({
            'id': gid,
            'first': g['date'].min(),
            'last': g['date'].max(),
            'n_points': actual,
            'gaps': gaps,
        })
    qs = pd.DataFrame(stats).sort_values('n_points')
    return qs

qs = quality_checks(prepro_df)
display(qs.head(10))

print("IDs mit Lücken (gaps > 0):", (qs['gaps'] > 0).sum(), "von", len(qs))

In [ ]:
prepro_df.dtypes

In [ ]:
# TimeSeries bauen
series_by_id = {}
for gid, g in prepro_df.groupby("id"):
    ts = TimeSeries.from_dataframe(g, time_col="date", value_cols="demand", freq="D")
    ts = ts_fill_missing(ts, method=FILL_METHOD)
    series_by_id[gid] = ts

print(f"Anzahl Serien: {len(series_by_id)}")

In [ ]:
# Globale Cut-Daten
global_train_cut = get_global_train_cut(prepro_df, VAL_DAYS, TEST_DAYS)
print(f"Val days: {VAL_DAYS}, Test days: {TEST_DAYS}\n")
print(f"Trainingsdaten beginnen am:  {prepro_df['date'].min()}")
print(f"Globaler Trainingsdaten-Cut: {global_train_cut}")
print(f"Trainingsdaten enden am:     {global_train_cut}")
print(f"Validierung bis:             {global_train_cut + pd.Timedelta(days=VAL_DAYS)}")
print(f"Test bis:                    {global_train_cut + pd.Timedelta(days=VAL_DAYS + TEST_DAYS)}")

In [ ]:
# Hilfsfunktion: DataFrame → TimeSeries (mit Missing-Value-Füllung)
def _to_ts(g):
    ts = TimeSeries.from_dataframe(g, time_col="date", value_cols="demand", freq="D")
    if DARTS_HAS_MV_UTILS:
        try:
            ts = fill_missing_values(ts, fill="auto")
        except Exception:
            s = ts.pd_series().ffill().bfill()
            ts = TimeSeries.from_series(s, freq="D")
    else:
        s = ts.pd_series().ffill().bfill()
        ts = TimeSeries.from_series(s, freq="D")
    return ts

In [ ]:
# ----- Seasonality-Check -----
rows = []
for gid, g in prepro_df.groupby("id"):
    ts = _to_ts(g)
    train, _ = ts.split_after(global_train_cut)
    n = len(train)

    rec = {"id": gid, "n_train": n}
    for m in (7, 14, 28):
        L = 4*m                       # fixe Ziel-Obergrenze (global konsistent)
        eff_max_lag = min(n-1, L)
        if eff_max_lag < m:
            is_seas, eff_m = (False, None)   # oder np.nan, wenn du „nicht beurteilbar“ markieren willst
        else:
            try:
                is_seas, eff_m = check_seasonality(train, m=m, max_lag=eff_max_lag)
            except Exception:
                is_seas, eff_m = (False, None)

        rec[f"is_seasonal_m{m}"] = bool(is_seas)
        rec[f"eff_m{m}"] = int(eff_m) if eff_m is not None else None
        rec[f"eff_max_lag_m{m}"] = eff_max_lag
    rows.append(rec)

flags = pd.DataFrame(rows)

# Speichern
fname = f"seasonality_flags_{DATASET.lower()}_traincut_val{VAL_DAYS}_test{TEST_DAYS}.csv"
out_csv = OUT_DIR / fname
flags.to_csv(out_csv, index=False)
print("Seasonality-Flags gespeichert nach:", out_csv)

# Kurze Zusammenfassung
for m in M_LIST:
    total = len(flags)
    n_true = flags[f"is_seasonal_m{m}"].sum()
    print(f"m={m}: {n_true}/{total} Serien (~{100*n_true/total:.1f}%) mit signifikanter Saisonalität")

display(flags.head())

### Visualisierungs Ideen

In [ ]:
# === Setup für Plots ===
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATASET = "bakery"  # ggf. anpassen
M_LIST  = [7,14,28] # muss zu deinen Spalten in `flags` passen

PLOT_DIR = Path.cwd().parents[1] / "outputs" / "eda" / DATASET / "figs"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Long-Format bauen
records = []
for m in M_LIST:
    dfm = flags[['id','n_train',f'is_seasonal_m{m}',f'eff_m{m}',f'eff_max_lag_m{m}']].copy()
    dfm['m'] = m
    dfm = dfm.rename(columns={f'is_seasonal_m{m}':'is_seasonal',
                              f'eff_m{m}':'eff_m',
                              f'eff_max_lag_m{m}':'eff_max_lag'})
    records.append(dfm)
long = pd.concat(records, ignore_index=True)

In [ ]:
share = (long.groupby('m')['is_seasonal']
              .value_counts(normalize=True)
              .rename('share')
              .reset_index())

fig, ax = plt.subplots(figsize=(6,4))
for i, m in enumerate(sorted(M_LIST)):
    s = share[share['m']==m]
    yes = float(s[s['is_seasonal']==True]['share']) if any((s['is_seasonal']==True)) else 0.0
    ax.bar(i, 1.0, label=None)             # 100%
    ax.bar(i, yes, label=None)             # Anteil True übermalen
    ax.text(i, 0.5, f"{yes*100:.0f}%", ha='center', va='center', color='white', weight='bold')
ax.set_xticks(range(len(M_LIST))); ax.set_xticklabels([f"m={m}" for m in M_LIST])
ax.set_ylim(0,1); ax.set_ylabel("Anteil (True)")
ax.set_title("Saisonalität erkannt (Anteil je m)")
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_share_per_m.png", dpi=150)
plt.show()


In [ ]:
mat = (long.pivot(index='id', columns='m', values='is_seasonal')
           .reindex(columns=sorted(M_LIST)))
# IDs optional nach n_train sortieren:
id_order = (long.groupby('id')['n_train'].max()
                 .sort_values(ascending=False).index)
mat = mat.loc[id_order]
mat_int = mat.astype(float)  # True=1.0, False=0.0, NaN bleibt NaN

fig, ax = plt.subplots(figsize=(6, max(4, len(mat_int)/10)))
im = ax.imshow(mat_int.values, aspect='auto')
ax.set_xticks(range(len(M_LIST))); ax.set_xticklabels([f"m={m}" for m in M_LIST])
ax.set_yticks([])  # viele IDs -> Achse aus
ax.set_title("Saisonalität je ID und m (True=1, False=0)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_heatmap_id_by_m.png", dpi=150)
plt.show()

In [ ]:
# count id x m combinations for all True, all False, mixed
def count_id_m_combinations(df):
    counts = {'all_true': 0, 'all_false': 0, 'mixed': 0}
    for gid, g in df.groupby('id'):
        vals = g['is_seasonal'].dropna().unique()
        if len(vals) == 1:
            if vals[0] == True:
                counts['all_true'] += 1
            else:
                counts['all_false'] += 1
        elif len(vals) > 1:
            counts['mixed'] += 1
    return counts

counts = count_id_m_combinations(long)
print("Anzahl IDs mit:")
for k, v in counts.items():
    print(f"  {k}: {v}")

In [ ]:
# list all id x m combis that are seasonal
seasonal_combis = long[long['is_seasonal'] == True][['id', 'm', 'eff_m', 'n_train']]
print(f"Anzahl saisonaler ID x m-Kombis: {len(seasonal_combis)}")
display(seasonal_combis.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for m in sorted(M_LIST):
    dfm = long[long['m']==m]
    # Jitter auf y, damit Punkte pro m nicht exakt übereinander liegen
    y = np.full(len(dfm), m) + (np.random.rand(len(dfm))-0.5)*0.3
    ax.scatter(dfm['n_train'], y, s=12, alpha=0.6,
               label=f"m={m}")
ax.set_xlabel("n_train")
ax.set_ylabel("m (leicht gejittert)")
ax.set_title("Train-Länge je ID über m")
ax.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_scatter_ntrain_by_m.png", dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(M_LIST), figsize=(4*len(M_LIST),4), sharey=True)
if len(M_LIST)==1:
    axes = [axes]
for ax, m in zip(axes, sorted(M_LIST)):
    dfm = long[long['m']==m].copy()
    dfm['flag'] = np.where(dfm['is_seasonal'], 'seasonal', 'non-seasonal')
    groups = [dfm[dfm['flag']=='non-seasonal']['n_train'],
              dfm[dfm['flag']=='seasonal']['n_train']]
    ax.boxplot(groups, labels=['non','seasonal'])
    ax.set_title(f"m={m}")
    ax.set_xlabel("Flag"); ax.set_ylabel("n_train")
fig.suptitle("Train-Länge vs. Saisonalität")
plt.tight_layout()
plt.savefig(PLOT_DIR / "seasonality_boxplot_ntrain_by_flag.png", dpi=150)
plt.show()